# Data Integration and Aggregation

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [2]:
spark = SparkSession.builder.appName('DataIntegration&Aggregation').getOrCreate()

26/03/11 06:13:51 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [3]:
hdfs_path = '/data/olist/'

customer_df = spark.read.csv(hdfs_path + 'olist_customers_dataset.csv', header=True, inferSchema=True)
orders_df = spark.read.csv(hdfs_path + 'olist_orders_dataset.csv', header=True, inferSchema=True)
order_items_df = spark.read.csv(hdfs_path + 'olist_order_items_dataset.csv' , header=True, inferSchema=True)
payments_df = spark.read.csv(hdfs_path + 'olist_order_payments_dataset.csv' , header=True, inferSchema=True)
reviews_df = spark.read.csv(hdfs_path + 'olist_order_reviews_dataset.csv' , header=True, inferSchema=True)
products_df = spark.read.csv(hdfs_path + 'olist_products_dataset.csv',  header=True, inferSchema=True)
sellers_df = spark.read.csv(hdfs_path + 'olist_sellers_dataset.csv' , header=True, inferSchema=True)
geolocation_df = spark.read.csv(hdfs_path + 'olist_geolocation_dataset.csv' , header=True, inferSchema=True)
product_category_df = spark.read.csv(hdfs_path + 'product_category_name_translation.csv',header=True, inferSchema=True )

In [5]:
# caching the Dataframes that are always used or Frequently used Data for Better performance

orders_df.cache()
customer_df.cache()
order_items_df.cache()

DataFrame[order_id: string, order_item_id: int, product_id: string, seller_id: string, shipping_limit_date: timestamp, price: double, freight_value: double]

In [7]:
# Joining all the Dataframes 

order_items_joined_df = orders_df.join(order_items_df, 'order_id', 'inner')
order_items_product_joined_df = order_items_joined_df.join(products_df, 'product_id', 'inner')
order_items_product_sellers_joined_df = order_items_product_joined_df.join(sellers_df, 'seller_id', 'inner')
full_orders_df = order_items_product_sellers_joined_df.join(customer_df, 'customer_id', 'inner')

# Geolocation joining using left join
full_orders_df = full_orders_df.join(geolocation_df, full_orders_df.customer_zip_code_prefix == geolocation_df.geolocation_zip_code_prefix, 'left')

# reviews_df joining using left join
full_orders_df = full_orders_df.join(reviews_df, 'order_id', 'left')

# payment_df joining using left join
full_orders_df = full_orders_df.join(payments_df, 'order_id', 'left')

In [8]:
full_orders_df.cache()

26/03/10 09:10:11 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


DataFrame[order_id: string, customer_id: string, seller_id: string, product_id: string, order_status: string, order_purchase_timestamp: timestamp, order_approved_at: timestamp, order_delivered_carrier_date: timestamp, order_delivered_customer_date: timestamp, order_estimated_delivery_date: timestamp, order_item_id: int, shipping_limit_date: timestamp, price: double, freight_value: double, product_category_name: string, product_name_lenght: int, product_description_lenght: int, product_photos_qty: int, product_weight_g: int, product_length_cm: int, product_height_cm: int, product_width_cm: int, seller_zip_code_prefix: int, seller_city: string, seller_state: string, customer_unique_id: string, customer_zip_code_prefix: int, customer_city: string, customer_state: string, geolocation_zip_code_prefix: int, geolocation_lat: double, geolocation_lng: double, geolocation_city: string, geolocation_state: string, review_id: string, review_score: string, review_comment_title: string, review_commen

In [9]:
# Total Revenues Per Seller

seller_revenue_df = full_orders_df.groupBy('seller_id').agg(F.sum('price').alias('total_revenue'))

In [10]:
# This will take a long time in first run since we're caching the full_orders_df
seller_revenue_df.show()

+--------------------+--------------------+
|           seller_id|       total_revenue|
+--------------------+--------------------+
|b76dba6c951ab00dc...|  302582.65999999916|
|7a67c85e85bb2ce85...|2.0312794890000023E7|
|d2374cbcbb3ca4ab1...|   3375517.550000011|
|c8b0e2b0a7095e5d8...|  1573840.0600000022|
|994f04b3718c2bab3...|   661633.6000000022|
|88460e8ebdecbfecb...|   6320713.500000003|
|cca3071e3e9bb7d12...| 1.011593426999999E7|
|d13e50eaa47b4cbe9...|  1210454.8000000005|
|92eb0f42c21942b65...|   912998.0599999975|
|f3b80352b986ab4d1...|  1435817.2000000011|
|02f623a8eb246f3c5...|            957411.0|
|7040e82f899a04d1b...|  1517013.4000000043|
|e9779976487b77c6d...|   6293200.690000005|
|a1043bafd471dff53...| 1.766267597999997E7|
|ccc4bbb5f32a6ab2b...|   9748464.180000003|
|392e0502231ae2f8b...|   251834.3999999999|
|98dac6635aee4995d...|   946635.9299999672|
|7f35f9daf223da737...|             44820.0|
|d263fa444c1504a75...|             25536.0|
|93dc87703c046b603...|          

In [12]:
# Total Orders Per Customer 

total_orders_per_customer = full_orders_df.groupBy('customer_id').agg(F.count('order_id').alias('total_orders'))
total_orders_per_customer.show(5)

+--------------------+------------+
|         customer_id|total_orders|
+--------------------+------------+
|9c8410ae611b0c9fd...|          60|
|1e49d70b227359f77...|          64|
|2e3160c0e701d9d91...|          83|
|423b14adf6348b595...|         166|
|f81bb64a1e672e6cb...|         454|
+--------------------+------------+
only showing top 5 rows



In [13]:
# Average Review Score Per Seller

average_review_score_per_seller = full_orders_df.groupBy('seller_id').agg(F.avg('review_score').alias('average_score'))
average_review_score_per_seller.show(5)

+--------------------+------------------+
|           seller_id|     average_score|
+--------------------+------------------+
|b76dba6c951ab00dc...| 4.219273172723561|
|7a67c85e85bb2ce85...| 4.258920734844587|
|d2374cbcbb3ca4ab1...|  3.71899382437114|
|c8b0e2b0a7095e5d8...|3.7818383167220375|
|994f04b3718c2bab3...| 4.019792907541464|
+--------------------+------------------+
only showing top 5 rows



In [14]:
# Most Sold Products ( Top 10 )

most_sold_products = full_orders_df.groupBy('product_id').agg(F.count('order_purchase_timestamp').alias('most_sold')).orderBy(F.col('most_sold'), ascending=False)
most_sold_products.show(10)

+--------------------+---------+
|          product_id|most_sold|
+--------------------+---------+
|aca2eb7d00ea1a7b8...|    86740|
|422879e10f4668299...|    81110|
|99a4788cb24856965...|    78775|
|389d119b48cf3043d...|    60248|
|d1c427060a0f73f6b...|    59274|
|368c6c730842d7801...|    58358|
|53759a2ecddad2bb8...|    52654|
|53b36df67ebb7c415...|    52105|
|154e7e31ebfa09220...|    42700|
|3dd2a17168ec895c7...|    40787|
+--------------------+---------+
only showing top 10 rows



In [15]:
# Top Customers By Spending 

top_customers_by_spending = full_orders_df.groupBy('customer_id').agg(F.count('order_purchase_timestamp').alias('customer_spends')).orderBy(F.col('customer_spends'), ascending=False)
top_customers_by_spending.show(5)


+--------------------+---------------+
|         customer_id|customer_spends|
+--------------------+---------------+
|351e40989da90e704...|          11427|
|50920f8cd0681fd86...|          10752|
|9b43e2a62de9bab3a...|           8556|
|270c23a11d024a44c...|           8001|
|5c87184371002d49e...|           6876|
+--------------------+---------------+
only showing top 5 rows



# Optimized Joins for Data Integration

In [4]:
# Joining all the Dataframes 

order_items_joined_df = orders_df.join(order_items_df, 'order_id', 'inner')
order_items_product_joined_df = order_items_joined_df.join(products_df, 'product_id', 'inner')
order_items_product_sellers_joined_df = order_items_product_joined_df.join(F.broadcast(sellers_df), 'seller_id', 'inner')
full_orders_df = order_items_product_sellers_joined_df.join(customer_df, 'customer_id', 'inner')

# Geolocation joining using left join
full_orders_df = full_orders_df.join(F.broadcast(geolocation_df), full_orders_df.customer_zip_code_prefix == geolocation_df.geolocation_zip_code_prefix, 'left')

# reviews_df joining using left join
full_orders_df = full_orders_df.join(F.broadcast(reviews_df), 'order_id', 'left')

# payment_df joining using left join
full_orders_df = full_orders_df.join(payments_df, 'order_id', 'left')

full_orders_df.cache()

26/03/11 06:14:44 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


DataFrame[order_id: string, customer_id: string, seller_id: string, product_id: string, order_status: string, order_purchase_timestamp: timestamp, order_approved_at: timestamp, order_delivered_carrier_date: timestamp, order_delivered_customer_date: timestamp, order_estimated_delivery_date: timestamp, order_item_id: int, shipping_limit_date: timestamp, price: double, freight_value: double, product_category_name: string, product_name_lenght: int, product_description_lenght: int, product_photos_qty: int, product_weight_g: int, product_length_cm: int, product_height_cm: int, product_width_cm: int, seller_zip_code_prefix: int, seller_city: string, seller_state: string, customer_unique_id: string, customer_zip_code_prefix: int, customer_city: string, customer_state: string, geolocation_zip_code_prefix: int, geolocation_lat: double, geolocation_lng: double, geolocation_city: string, geolocation_state: string, review_id: string, review_score: string, review_comment_title: string, review_commen

# Window Function and Ranking

In [7]:
from pyspark.sql.window import Window

In [8]:
# Rank Top Selling Products Per Seller
# Dense Rank for Sellers Based on Revenue


window_spec = Window.partitionBy('seller_id').orderBy(F.desc('price'))

In [9]:
# Rank Top Selling Products Per Seller

top_seller_products_df = full_orders_df.withColumn('rank', F.rank().over(window_spec))\
.filter(F.col('rank') <= 5)

top_seller_products_df.select('seller_id', 'price', 'rank').show()

+--------------------+-----+----+
|           seller_id|price|rank|
+--------------------+-----+----+
|0015a82c2db000af6...|895.0|   1|
|0015a82c2db000af6...|895.0|   1|
|0015a82c2db000af6...|895.0|   1|
|0015a82c2db000af6...|895.0|   1|
|0015a82c2db000af6...|895.0|   1|
|0015a82c2db000af6...|895.0|   1|
|0015a82c2db000af6...|895.0|   1|
|0015a82c2db000af6...|895.0|   1|
|0015a82c2db000af6...|895.0|   1|
|0015a82c2db000af6...|895.0|   1|
|0015a82c2db000af6...|895.0|   1|
|0015a82c2db000af6...|895.0|   1|
|0015a82c2db000af6...|895.0|   1|
|0015a82c2db000af6...|895.0|   1|
|0015a82c2db000af6...|895.0|   1|
|0015a82c2db000af6...|895.0|   1|
|0015a82c2db000af6...|895.0|   1|
|0015a82c2db000af6...|895.0|   1|
|0015a82c2db000af6...|895.0|   1|
|0015a82c2db000af6...|895.0|   1|
+--------------------+-----+----+
only showing top 20 rows



In [11]:
# Dense Rank for Sellers Based on Revenue

dense_rank_for_sellers_df = full_orders_df.withColumn('dense_rank', F.dense_rank().over(window_spec)).filter(F.col('dense_rank')<=3)

dense_rank_for_sellers_df.select('seller_id', 'price', 'dense_rank').show()

+--------------------+-----+----------+
|           seller_id|price|dense_rank|
+--------------------+-----+----------+
|0015a82c2db000af6...|895.0|         1|
|0015a82c2db000af6...|895.0|         1|
|0015a82c2db000af6...|895.0|         1|
|0015a82c2db000af6...|895.0|         1|
|0015a82c2db000af6...|895.0|         1|
|0015a82c2db000af6...|895.0|         1|
|0015a82c2db000af6...|895.0|         1|
|0015a82c2db000af6...|895.0|         1|
|0015a82c2db000af6...|895.0|         1|
|0015a82c2db000af6...|895.0|         1|
|0015a82c2db000af6...|895.0|         1|
|0015a82c2db000af6...|895.0|         1|
|0015a82c2db000af6...|895.0|         1|
|0015a82c2db000af6...|895.0|         1|
|0015a82c2db000af6...|895.0|         1|
|0015a82c2db000af6...|895.0|         1|
|0015a82c2db000af6...|895.0|         1|
|0015a82c2db000af6...|895.0|         1|
|0015a82c2db000af6...|895.0|         1|
|0015a82c2db000af6...|895.0|         1|
+--------------------+-----+----------+
only showing top 20 rows



# Advance Aggregation and Enrichment

In [5]:
# Total Revenue & Average Order Value (AOV) per Customer

customer_spending_df = full_orders_df.groupBy('customer_id')\
.agg(
    F.count('order_id').alias('total_orders'),
    F.sum('price').alias('total_spent'),
    F.round(F.avg('price'), 2).alias('AOV')
).orderBy(F.desc('total_spent'))

customer_spending_df.show()

+--------------------+------------+------------------+-------+
|         customer_id|total_orders|       total_spent|    AOV|
+--------------------+------------+------------------+-------+
|d3e82ccec3cb5f956...|        6876|         6662844.0|  969.0|
|df55c14d1476a9a34...|         743|         3565657.0| 4799.0|
|fe5113a38e3575c04...|        2292|         3293604.0| 1437.0|
|ec5b2ba62e5743423...|        1428|         2556120.0| 1790.0|
|63b964e79dee32a35...|        6072|         2501664.0|  412.0|
|46bb3c0b1a65c8399...|         748|         2336752.0| 3124.0|
|05455dfa7cd02f13d...|        2184| 2160194.400000087|  989.1|
|3690e975641f01bd0...|         802|         2124498.0| 2649.0|
|349509b216bd5ec11...|         743|         1923627.0| 2589.0|
|695476b5848d64ba0...|         687|1820543.1299999943|2649.99|
|73236a0796f53d60d...|         832|         1755520.0| 2110.0|
|cc803a2c412833101...|         762|         1676400.0| 2200.0|
|1ff773612ab8934db...|        5820|1658641.7999999512| 

In [14]:
# Seller Performance Metrics (Revenue, Average Review, Order Count)


seller_performance_df = full_orders_df.groupBy('seller_id')\
.agg(
    F.count('order_id').alias('total_orders'),
    F.sum('price').alias('total_revenue'),
    F.round(F.avg('review_score'), 2).alias('average_review_score'),
    F.round(F.stddev('price'), 2).alias('price_variability')
).orderBy(F.desc('total_revenue'))

seller_performance_df.show()

+--------------------+------------+--------------------+--------------------+-----------------+
|           seller_id|total_orders|       total_revenue|average_review_score|price_variability|
+--------------------+------------+--------------------+--------------------+-----------------+
|4869f7a5dfa277a7d...|      184587| 3.613871731999314E7|                4.09|           111.65|
|53243585a1d6dc264...|       54514|3.4291592950000696E7|                4.12|           499.65|
|4a3ca9315b744ce9f...|      330661| 3.375957084001202E7|                3.77|            59.37|
|7c67e1448b00f6e96...|      233306|3.2282321790021457E7|                3.42|            50.39|
|fa1c13f2614d7b5c4...|       87686|3.0139386310006626E7|                4.38|            307.7|
|da8622b14eb17ae28...|      264433| 2.985766973003611E7|                3.98|            72.92|
|7e93a43ef30c4f03f...|       50226| 2.631570630000355E7|                4.15|           377.24|
|1025f0e2d44d7041d...|      229587|2.293

In [15]:
# Product Popularity Metrics

product_metrics_df = full_orders_df.groupBy('product_id')\
.agg(
    F.count('order_id').alias('total_sales'),
    F.sum('price').alias('total_revenue'),
    F.round(F.avg('price'), 2).alias('average_price'),
    F.round(F.stddev('price'), 2).alias('price_volatility'),
    F.collect_set('seller_id').alias('unique_sellers')
).orderBy(F.desc('total_sales'))

product_metrics_df.show()

+--------------------+-----------+------------------+-------------+----------------+--------------------+
|          product_id|total_sales|     total_revenue|average_price|price_volatility|      unique_sellers|
+--------------------+-----------+------------------+-------------+----------------+--------------------+
|aca2eb7d00ea1a7b8...|      86740| 6164630.299996213|        71.07|            3.17|[955fee9216a65b61...|
|422879e10f4668299...|      81110| 4442791.509997526|        54.77|            4.46|[1f50f920176fa81d...|
|99a4788cb24856965...|      78775| 6921762.709996367|        87.87|            4.08|[4a3ca9315b744ce9...|
|389d119b48cf3043d...|      60248|3280533.1299988106|        54.45|            4.37|[1f50f920176fa81d...|
|d1c427060a0f73f6b...|      59274| 8220103.330002412|       138.68|           16.58|[a1043bafd471dff5...|
|368c6c730842d7801...|      58358| 3181698.899999002|        54.52|            4.59|[1f50f920176fa81d...|
|53759a2ecddad2bb8...|      52654|2893017.4999

In [11]:
full_orders_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: integer (nullable = true)
 |-- product_description_lenght: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (null

In [17]:
# Monthly Revenue and Order Count Trend 

monthly_revenue_trend_df = full_orders_df.withColumn('Month', F.month('order_purchase_timestamp')).groupBy('seller_id', 'Month')\
.agg(
    F.count('order_id').alias('total_orders'),
    F.sum('price').alias('total_revenue'),
    F.round(F.avg('price'), 2).alias('avg_order_value'),
    F.min('price').alias('min_order_value'),
    F.max('price').alias('max_order_value')
).orderBy(F.desc('total_orders'))

monthly_revenue_trend_df.show()

+--------------------+-----+------------+------------------+---------------+---------------+---------------+
|           seller_id|Month|total_orders|     total_revenue|avg_order_value|min_order_value|max_order_value|
+--------------------+-----+------------+------------------+---------------+---------------+---------------+
|1f50f920176fa81da...|   11|       54638| 2890335.729999636|           52.9|           49.0|           59.9|
|6560211a19b47992c...|    8|       40682|         2537196.0|          62.37|           21.0|          249.0|
|4a3ca9315b744ce9f...|    5|       39829|4173609.3799987542|         104.79|           41.8|          305.0|
|955fee9216a65b617...|    5|       39511|4047132.5399993127|         102.43|           13.6|          750.0|
|955fee9216a65b617...|    4|       38987| 3025763.089999661|          77.61|           12.0|          500.0|
|7c67e1448b00f6e96...|    3|       37731| 4909587.220000462|         130.12|          29.99|         239.99|
|4a3ca9315b744ce9f.

In [25]:
# Customer Retention Analysis ( First & Last Order)

window_specs = Window.partitionBy('customer_id').orderBy('order_purchase_timestamp')

customer_retention_df = full_orders_df.withColumn('first_order_date', F.first('order_purchase_timestamp').over(window_specs))\
.withColumn('last_order_date', F.last('order_purchase_timestamp').over(window_specs))\
.groupBy('customer_id', 'first_order_date', 'last_order_date')\
.agg(
    F.count('order_id').alias('total_orders'),
    F.round(F.avg('price'), 2).alias('aov')
).orderBy(F.desc('total_orders'))

customer_retention_df.show()

+--------------------+-------------------+-------------------+------------+------+
|         customer_id|   first_order_date|    last_order_date|total_orders|   aov|
+--------------------+-------------------+-------------------+------------+------+
|351e40989da90e704...|2017-07-13 10:42:37|2017-07-13 10:42:37|       11427| 85.99|
|50920f8cd0681fd86...|2018-01-27 11:28:32|2018-01-27 11:28:32|       10752| 43.82|
|9b43e2a62de9bab3a...|2017-05-25 22:27:50|2017-05-25 22:27:50|        8556|  26.4|
|270c23a11d024a44c...|2017-08-08 20:26:31|2017-08-08 20:26:31|        8001| 36.59|
|5c87184371002d49e...|2018-01-05 19:15:37|2018-01-05 19:15:37|        6876| 12.49|
|d3e82ccec3cb5f956...|2017-03-18 14:28:34|2017-03-18 14:28:34|        6876| 969.0|
|d5f2b3f597c7ccafb...|2017-12-13 14:21:15|2017-12-13 14:21:15|        6706|  59.0|
|c2f18647725395af4...|2018-03-06 19:21:47|2018-03-06 19:21:47|        6612|  34.9|
|24e7dc2ff8c071263...|2017-11-24 16:16:45|2017-11-24 16:16:45|        6597|  59.2|
|7bb

In [29]:
full_orders_df.groupBy('customer_id', 'order_purchase_timestamp').count().show()   

+--------------------+------------------------+-----+
|         customer_id|order_purchase_timestamp|count|
+--------------------+------------------------+-----+
|ea2faf6b4dcc7a219...|     2018-03-19 02:21:39|  112|
|1f0297c67e5e3106b...|     2018-01-08 10:33:19|   43|
|d5551343ca45deaab...|     2018-08-16 15:15:59|   34|
|5b79686b44ff3de0d...|     2018-04-01 13:54:37|   38|
|909e605a5ecf709a3...|     2017-12-11 01:05:34|   66|
|c660c3abf6e4e1ad1...|     2017-09-20 22:05:03|   45|
|cb26ed4a22f051e22...|     2017-05-28 11:40:24|   68|
|9131a1850f1153939...|     2018-06-28 19:34:59|  218|
|ede0af891a1bd75e5...|     2017-02-23 15:07:15|  286|
|37c9f4ed2c9875607...|     2018-08-08 12:52:01|   14|
|2bdd0a7c4c3c8942e...|     2017-07-07 21:09:46|   46|
|513df1f8363b2dd26...|     2017-05-30 15:43:45|  128|
|53cbe14b65401ddcd...|     2018-03-18 12:16:38|   63|
|aa7210736aad65a84...|     2018-03-29 08:49:38|   58|
|97979d55a21d30d70...|     2017-05-21 19:55:20|  286|
|636e15840ab051faa...|     2

# Extended Enrichment

In [33]:
# Order Status Flags

full_orders_df = full_orders_df.withColumn('is_delivered', F.when(F.col('order_status')== 'delivered', F.lit(1)).otherwise(F.lit(0)))\
.withColumn('is_cancelled', F.when(F.col('order_status')=='canceled', F.lit(1)).otherwise(F.lit(0)))

full_orders_df.where(full_orders_df['order_status']=='canceled').select('order_status', 'is_delivered', 'is_cancelled').show()

+------------+------------+------------+
|order_status|is_delivered|is_cancelled|
+------------+------------+------------+
|    canceled|           0|           1|
|    canceled|           0|           1|
|    canceled|           0|           1|
|    canceled|           0|           1|
|    canceled|           0|           1|
|    canceled|           0|           1|
|    canceled|           0|           1|
|    canceled|           0|           1|
|    canceled|           0|           1|
|    canceled|           0|           1|
|    canceled|           0|           1|
|    canceled|           0|           1|
|    canceled|           0|           1|
|    canceled|           0|           1|
|    canceled|           0|           1|
|    canceled|           0|           1|
|    canceled|           0|           1|
|    canceled|           0|           1|
|    canceled|           0|           1|
|    canceled|           0|           1|
+------------+------------+------------+
only showing top

In [8]:
# Order Revenue Calculation

full_orders_df = full_orders_df.withColumn('order_revenue', F.col('price') + F.col('freight_value'))
full_orders_df.select('price', 'freight_value', 'order_revenue').show()

+-----+-------------+-------------+
|price|freight_value|order_revenue|
+-----+-------------+-------------+
|29.99|         8.72|        38.71|
|29.99|         8.72|        38.71|
|29.99|         8.72|        38.71|
|29.99|         8.72|        38.71|
|29.99|         8.72|        38.71|
|29.99|         8.72|        38.71|
|29.99|         8.72|        38.71|
|29.99|         8.72|        38.71|
|29.99|         8.72|        38.71|
|29.99|         8.72|        38.71|
|29.99|         8.72|        38.71|
|29.99|         8.72|        38.71|
|29.99|         8.72|        38.71|
|29.99|         8.72|        38.71|
|29.99|         8.72|        38.71|
|29.99|         8.72|        38.71|
|29.99|         8.72|        38.71|
|29.99|         8.72|        38.71|
|29.99|         8.72|        38.71|
|29.99|         8.72|        38.71|
+-----+-------------+-------------+
only showing top 20 rows



In [6]:
# Customer Segmentation based on spending

customer_spending_df = customer_spending_df.withColumn(
    'customer_segment', 
    F.when(F.col('AOV') >= 1200, 'High-Value')
    .when((F.col('AOV') >= 500) & (F.col('AOV') <= 1200), 'Medium-Value')
    .otherwise('Low-Value')
)

customer_spending_df.show()

+--------------------+------------+------------------+-------+----------------+
|         customer_id|total_orders|       total_spent|    AOV|customer_segment|
+--------------------+------------+------------------+-------+----------------+
|d3e82ccec3cb5f956...|        6876|         6662844.0|  969.0|    Medium-Value|
|df55c14d1476a9a34...|         743|         3565657.0| 4799.0|      High-Value|
|fe5113a38e3575c04...|        2292|         3293604.0| 1437.0|      High-Value|
|ec5b2ba62e5743423...|        1428|         2556120.0| 1790.0|      High-Value|
|63b964e79dee32a35...|        6072|         2501664.0|  412.0|       Low-Value|
|46bb3c0b1a65c8399...|         748|         2336752.0| 3124.0|      High-Value|
|05455dfa7cd02f13d...|        2184| 2160194.400000087|  989.1|    Medium-Value|
|3690e975641f01bd0...|         802|         2124498.0| 2649.0|      High-Value|
|349509b216bd5ec11...|         743|         1923627.0| 2589.0|      High-Value|
|695476b5848d64ba0...|         687|18205

In [7]:
full_orders_df = full_orders_df.join(customer_spending_df.select('customer_id', 'customer_segment'), 'customer_id', 'left')

In [8]:
full_orders_df.groupBy('customer_id', 'customer_segment').count().orderBy(F.desc('count')).show()

+--------------------+----------------+-----+
|         customer_id|customer_segment|count|
+--------------------+----------------+-----+
|351e40989da90e704...|       Low-Value|11427|
|50920f8cd0681fd86...|       Low-Value|10752|
|9b43e2a62de9bab3a...|       Low-Value| 8556|
|270c23a11d024a44c...|       Low-Value| 8001|
|5c87184371002d49e...|       Low-Value| 6876|
|d3e82ccec3cb5f956...|    Medium-Value| 6876|
|d5f2b3f597c7ccafb...|       Low-Value| 6706|
|c2f18647725395af4...|       Low-Value| 6612|
|24e7dc2ff8c071263...|       Low-Value| 6597|
|7bb57d182bdc11653...|       Low-Value| 6258|
|63b964e79dee32a35...|       Low-Value| 6072|
|d22f25a9fadfb1abb...|       Low-Value| 6072|
|1ff773612ab8934db...|       Low-Value| 5820|
|13aa59158da63ba0e...|       Low-Value| 5206|
|78fc46047c4a639e8...|       Low-Value| 5200|
|dd3f1762eb601f41c...|       Low-Value| 4992|
|a193aa8d905b8e246...|       Low-Value| 4896|
|9eb3d566e87289dcb...|       Low-Value| 4872|
|2ba91e12e5e4c9f56...|       Low-V

In [11]:
# Hourly Order Distribution

full_orders_df = full_orders_df.withColumn('hour_of_day', F.expr('hour(order_purchase_timestamp)'))

In [17]:
full_orders_df.select('order_purchase_timestamp','hour_of_day').show()

+------------------------+-----------+
|order_purchase_timestamp|hour_of_day|
+------------------------+-----------+
|     2017-10-02 10:56:33|         10|
|     2017-10-02 10:56:33|         10|
|     2017-10-02 10:56:33|         10|
|     2017-10-02 10:56:33|         10|
|     2017-10-02 10:56:33|         10|
|     2017-10-02 10:56:33|         10|
|     2017-10-02 10:56:33|         10|
|     2017-10-02 10:56:33|         10|
|     2017-10-02 10:56:33|         10|
|     2017-10-02 10:56:33|         10|
|     2017-10-02 10:56:33|         10|
|     2017-10-02 10:56:33|         10|
|     2017-10-02 10:56:33|         10|
|     2017-10-02 10:56:33|         10|
|     2017-10-02 10:56:33|         10|
|     2017-10-02 10:56:33|         10|
|     2017-10-02 10:56:33|         10|
|     2017-10-02 10:56:33|         10|
|     2017-10-02 10:56:33|         10|
|     2017-10-02 10:56:33|         10|
+------------------------+-----------+
only showing top 20 rows



In [15]:
# Weekday vs Weekend Order

full_orders_df = full_orders_df.withColumn(
    "order_day_type",
    F.when(
        F.dayofweek(F.col("order_purchase_timestamp")).isin(1, 7),
        F.lit("Weekend")
    ).otherwise(F.lit("Weekday"))
)

In [18]:
full_orders_df.select('order_purchase_timestamp','order_day_type').show()

+------------------------+--------------+
|order_purchase_timestamp|order_day_type|
+------------------------+--------------+
|     2017-10-02 10:56:33|       Weekday|
|     2017-10-02 10:56:33|       Weekday|
|     2017-10-02 10:56:33|       Weekday|
|     2017-10-02 10:56:33|       Weekday|
|     2017-10-02 10:56:33|       Weekday|
|     2017-10-02 10:56:33|       Weekday|
|     2017-10-02 10:56:33|       Weekday|
|     2017-10-02 10:56:33|       Weekday|
|     2017-10-02 10:56:33|       Weekday|
|     2017-10-02 10:56:33|       Weekday|
|     2017-10-02 10:56:33|       Weekday|
|     2017-10-02 10:56:33|       Weekday|
|     2017-10-02 10:56:33|       Weekday|
|     2017-10-02 10:56:33|       Weekday|
|     2017-10-02 10:56:33|       Weekday|
|     2017-10-02 10:56:33|       Weekday|
|     2017-10-02 10:56:33|       Weekday|
|     2017-10-02 10:56:33|       Weekday|
|     2017-10-02 10:56:33|       Weekday|
|     2017-10-02 10:56:33|       Weekday|
+------------------------+--------

In [19]:
full_orders_df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: integer (nullable = true)
 |-- product_description_lenght: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (null

In [28]:
full_orders_df.groupBy('customer_id').agg(
  F.max('freight_value')
).show() 

+--------------------+------------------+
|         customer_id|max(freight_value)|
+--------------------+------------------+
|6c8dc19e9aa79a2c4...|             19.15|
|fab1a643ad6f3da34...|             25.63|
|d84f2788580b9d5e4...|             31.97|
|f0f97266247be0432...|              17.6|
|9801ac60f0291d180...|             18.33|
|04cef6b920c0d8f16...|             10.82|
|3f0f45bd49f790854...|              9.45|
|64247d2cff9b9b98b...|              12.8|
|30f523f7def36192e...|             57.09|
|8f4758b55faa41da5...|             16.11|
|2ef8398f62042a2b4...|             69.48|
|c79c8d356d92bb1b7...|             13.27|
|a78e85d7d80bdae62...|             21.05|
|a836f6725983cd799...|             10.96|
|ac9b518157bd24e32...|             17.65|
|4c49ffd00d17560e8...|             21.09|
|d927892e658a6af91...|              19.4|
|a8004a3d658be3bb2...|              7.39|
|e4c1b6d865406867c...|             11.85|
|9d1a96b181bc6f175...|             13.71|
+--------------------+------------

In [29]:
# freight_value --> low 30 below, med 30 >  <40, or 40 above high>

full_orders_df = full_orders_df.withColumn(
    'freight_level',
    F.when(F.col('freight_value')<30, F.lit('Low'))\
    .when((F.col('freight_value')>=30) & (F.col('freight_value')<40), F.lit('Med'))\
    .otherwise('High')
)

full_orders_df.select('freight_value', 'freight_level').orderBy(F.desc('freight_level')).show()

+-------------+-------------+
|freight_value|freight_level|
+-------------+-------------+
|        34.92|          Med|
|        30.53|          Med|
|        34.92|          Med|
|        30.53|          Med|
|        34.92|          Med|
|        30.53|          Med|
|        34.92|          Med|
|        30.53|          Med|
|        34.92|          Med|
|        30.53|          Med|
|        34.92|          Med|
|        30.53|          Med|
|        34.92|          Med|
|        30.53|          Med|
|        34.92|          Med|
|        30.53|          Med|
|        34.92|          Med|
|        30.53|          Med|
|        34.92|          Med|
|        30.53|          Med|
+-------------+-------------+
only showing top 20 rows



In [31]:
full_orders_df.where(full_orders_df['freight_value'] > 40).select('freight_value', 'freight_level').show()

+-------------+-------------+
|freight_value|freight_level|
+-------------+-------------+
|        77.45|         High|
|        77.45|         High|
|        77.45|         High|
|        77.45|         High|
|        77.45|         High|
|        77.45|         High|
|        77.45|         High|
|        77.45|         High|
|        77.45|         High|
|        77.45|         High|
|        77.45|         High|
|        77.45|         High|
|        77.45|         High|
|        77.45|         High|
|        77.45|         High|
|        77.45|         High|
|        77.45|         High|
|        77.45|         High|
|        77.45|         High|
|        77.45|         High|
+-------------+-------------+
only showing top 20 rows

